In [54]:
import os
import torch 
import pandas as pd
from torchvision import models 
from torchinfo import summary
from pathlib import Path
from pytorch_memlab import profile

from model import init_model
from torchvision.transforms.functional import pad, to_pil_image
from torchvision.io import read_image
from torchvision.utils import save_image


In [39]:
def get_padding(image, max_w, max_h):
    
    h, w  = image.shape[1], image.shape[2]
    h_padding = (max_w - w) / 2
    v_padding = (max_h - h) / 2
    l_pad = h_padding if h_padding % 1 == 0 else h_padding+0.5
    t_pad = v_padding if v_padding % 1 == 0 else v_padding+0.5
    r_pad = h_padding if h_padding % 1 == 0 else h_padding-0.5
    b_pad = v_padding if v_padding % 1 == 0 else v_padding-0.5
    
    padding = (int(l_pad), int(t_pad), int(r_pad), int(b_pad))
    
    return padding


In [3]:
ROOT_DIR = Path(os.path.dirname(os.path.abspath('')))
CHANNELS = 3
BATCH_SIZE = 16
DEVICE = "cuda:1"
OCT_PRESENCE = "Usando OCT"
DUAL_IMAGE = "Dual Image"
DATA_PATH = "../data.csv"
SUMMARY_PATH = "../model_summary.csv"
HISTORY_PATH = "../history_csv"
FT_SIZE = 24
OUTPUT_TAB = 5

In [13]:
data = pd.read_csv(DATA_PATH)
max_w_1 = 0 
max_h_1 = 0
max_w_2 = 0 
max_h_2 = 0

In [20]:
for index, row in data.iterrows():
    photo_1 = read_image(str(ROOT_DIR / row["photo_1"]))
    photo_2 = read_image(str(ROOT_DIR / row["photo_2"]))
    if photo_1.shape[1] > max_h_1:
        max_h_1 = photo_1.shape[1]
    if photo_1.shape[2] > max_w_1:
        max_w_1 = photo_1.shape[2]
        
    if photo_2.shape[1] > max_h_2:
        max_h_2 = photo_2.shape[1]
    if photo_2.shape[2] > max_w_2:
        max_w_2 = photo_2.shape[2]
    

In [41]:
max_h_1

3456

In [42]:
max_w_1

5184

In [49]:
max_h_2

5404

In [50]:
max_w_2

5436

In [57]:
for index, row in data.iterrows():
    photo_1 = read_image(str(ROOT_DIR / row["photo_1"]))
    photo_2 = read_image(str(ROOT_DIR / row["photo_2"]))
    
    photo_1_padded = pad(photo_1, get_padding(photo_1,max_w_1, max_h_1))
    photo_2_padded = pad(photo_2, get_padding(photo_2,max_w_2, max_h_2))
    
    new_path_1 = str(ROOT_DIR / row["photo_1"]).replace("imgs","imgs_padded")
    new_path_2 = str(ROOT_DIR / row["photo_2"]).replace("imgs","imgs_padded")
    os.makedirs(os.path.dirname(new_path_1), exist_ok=True)
   
    to_pil_image(photo_1_padded).save(new_path_1)
    to_pil_image(photo_2_padded).save(new_path_2)    
    